In [ ]:
import pandas as pd
import numpy as np

# Simulamos datos
df = pd.DataFrame({
    'ingreso': np.random.choice(['bajo', 'medio', 'alto'], size=1000),
    'excluido': np.random.choice([0, 1], size=1000, p=[0.85, 0.15])
})

# Calculamos WOE por grupo
woe_table = df.groupby('ingreso').apply(
    lambda g: pd.Series({
        'No_excluidos': (g['excluido'] == 0).sum(),
        'Excluidos': (g['excluido'] == 1).sum()
    })
)

# Convertimos a proporciones
woe_table['%No_excluidos'] = woe_table['No_excluidos'] / woe_table['No_excluidos'].sum()
woe_table['%Excluidos'] = woe_table['Excluidos'] / woe_table['Excluidos'].sum()

# Calculamos WOE
woe_table['WOE'] = np.log(woe_table['%No_excluidos'] / woe_table['%Excluidos'])

print(woe_table[['%No_excluidos', '%Excluidos', 'WOE']])


         %No_excluidos  %Excluidos       WOE
ingreso                                     
alto          0.338390    0.321678  0.050646
bajo          0.329055    0.356643 -0.080512
medio         0.332555    0.321678  0.033254


/tmp/ipython-input-3859754634.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  woe_table = df.groupby('ingreso').apply(


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import KBinsDiscretizer

# Simulamos datos
np.random.seed(42)
n = 1000
df = pd.DataFrame({
    'edad': np.random.randint(18, 70, size=n),
    'ingreso': np.random.normal(12000, 4000, size=n),
    'historial_pagos': np.random.choice(['bueno', 'regular', 'malo'], size=n, p=[0.6, 0.3, 0.1]),
    'educacion': np.random.choice(['básica', 'media', 'superior'], size=n),
    'dependientes': np.random.randint(0, 5, size=n),
    'default': np.random.choice([0, 1], size=n, p=[0.85, 0.15])
})

# Binning de variables numéricas
binning = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='quantile')
df['edad_bin'] = binning.fit_transform(df[['edad']]).astype(int)
df['ingreso_bin'] = binning.fit_transform(df[['ingreso']]).astype(int)
df['dependientes_bin'] = binning.fit_transform(df[['dependientes']]).astype(int)

# Variables a transformar
variables = ['edad_bin', 'ingreso_bin', 'historial_pagos', 'educacion', 'dependientes_bin']

# Función para calcular WOE
def calcular_woe(df, variable, target='default'):
    woe_df = df.groupby(variable).apply(
        lambda g: pd.Series({
            'No_default': (g[target] == 0).sum(),
            'Default': (g[target] == 1).sum()
        })
    )
    woe_df['%No_default'] = woe_df['No_default'] / woe_df['No_default'].sum()
    woe_df['%Default'] = woe_df['Default'] / woe_df['Default'].sum()
    woe_df['WOE'] = np.log((woe_df['%No_default'] + 1e-6) / (woe_df['%Default'] + 1e-6))  # Evita división por cero
    return woe_df[['WOE']]

# Aplicamos WOE a todas las variables
woe_dict = {}
for var in variables:
    woe_dict[var] = calcular_woe(df, var)

# Ejemplo: WOE para historial de pagos
print("WOE por variable:")
print(woe_dict['ingreso_bin'])

# Puedes mapear WOE al dataframe si deseas usarlo en el modelo
for var in variables:
    df[f'{var}_WOE'] = df[var].map(woe_dict[var]['WOE'])


WOE por variable:
                  WOE
ingreso_bin          
0            0.052714
1            0.126385
2           -0.376565
3            0.287503


/tmp/ipython-input-995707848.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  woe_df = df.groupby(variable).apply(
/tmp/ipython-input-995707848.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  woe_df = df.groupby(variable).apply(
/tmp/ipython-input-995707848.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pan

In [2]:
# Variables transformadas
X_woe = df[['edad_bin_WOE', 'ingreso_bin_WOE', 'historial_pagos_WOE',
            'educacion_WOE', 'dependientes_bin_WOE']]
y = df['default']


In [3]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_woe, y)

# Extraemos coeficientes
coefs = pd.Series(model.coef_[0], index=X_woe.columns)
intercept = model.intercept_[0]


In [4]:
logit = model.predict_log_proba(X_woe)[:, 1]  # log(p / (1 - p))


In [5]:
factor = 20 / np.log(2)
offset = 600
df['score'] = (offset - factor * logit).round(0)
df

,edad,ingreso,historial_pagos,educacion,dependientes,default,edad_bin,ingreso_bin,dependientes_bin,edad_bin_WOE,ingreso_bin_WOE,historial_pagos_WOE,educacion_WOE,dependientes_bin_WOE,score
0,56,5574.214719,malo,superior,2,1,3,0,2,0.114956,0.052714,-0.098382,0.113313,-0.102456,662.0
1,69,12813.854543,bueno,media,4,0,3,2,3,0.114956,-0.376565,0.061138,-0.263906,0.029255,647.0
2,46,8974.597019,bueno,superior,1,0,2,0,1,0.103395,0.052714,0.061138,0.113313,-0.049894,663.0
3,32,6310.985162,regular,superior,4,1,1,0,3,-0.069666,0.052714,-0.085981,0.113313,0.029255,660.0
4,60,9413.708463,regular,superior,0,0,3,0,0,0.114956,0.052714,-0.085981,0.113313,0.111816,664.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,60,13678.129776,bueno,superior,0,0,3,2,0,0.114956,-0.376565,0.061138,0.113313,0.111816,655.0
996,64,8450.031289,bueno,básica,3,0,3,0,3,0.114956,0.052714,0.061138,0.189463,0.029255,666.0
997,62,10250.166799,bueno,básica,4,0,3,1,3,0.114956,0.126385,0.061138,0.189463,0.029255,667.0
998,35,14889.525431,bueno,básica,4,0,1,3,3,-0.069666,0.287503,0.061138,0.189463,0.029255,668.0


In [7]:
from sklearn.metrics import roc_curve
import numpy as np

# Probabilidades predichas para la clase positiva
y_scores = model.predict_proba(X_woe)[:, 1]

# Curva ROC
fpr, tpr, thresholds = roc_curve(y, y_scores)

# Índice de Youden
youden_index = tpr - fpr
optimal_idx = np.argmax(youden_index)
optimal_threshold = thresholds[optimal_idx]

# Mostrar el umbral óptimo
print("Umbral óptimo (Youden):", optimal_threshold)
logit_opt = np.log(optimal_threshold / (1 - optimal_threshold))
score_opt = offset - factor * logit_opt
print("Score óptimo:", score_opt)

Umbral óptimo (Youden): 0.1268497205586782
Score óptimo: 655.662192775565


In [8]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

# Entrenar modelo logístico
model = LogisticRegression()
model.fit(X_woe, y)

# Extraer coeficientes y definir escala
coefs = pd.Series(model.coef_[0], index=X_woe.columns)
factor = 20 / np.log(2)  # ≈ 28.85 puntos por unidad de logit
offset = 600             # Score base

# Calcular score por variable
score_por_variable = pd.DataFrame()
for var in X_woe.columns:
    contrib_logit = coefs[var] * X_woe[var]
    score_por_variable[var] = -factor * contrib_logit

# Calcular score total
df['score_total'] = offset + score_por_variable.sum(axis=1)

# Visualizar resultados
print("Score por variable:")
print(score_por_variable.head())

print("\nScore total:")
print(df[['score_total']].head())


Score por variable:
   edad_bin_WOE  ingreso_bin_WOE  historial_pagos_WOE  educacion_WOE  \
0      2.020946         1.369644            -0.635477       2.760294   
1      2.020946        -9.784152             0.394905      -6.428740   
2      1.817705         1.369644             0.394905       2.760294   
3     -1.224749         1.369644            -0.555377       2.760294   
4      2.020946         1.369644            -0.555377       2.760294   

   dependientes_bin_WOE  
0             -1.055733  
1              0.301451  
2             -0.514120  
3              0.301451  
4              1.152180  

Score total:
   score_total
0   604.459674
1   586.504411
2   605.828429
3   602.651262
4   606.747687


In [9]:
print("Coeficientes del modelo:")
print(coefs)

print("Factor:", factor)

print("Ejemplo de score por variable:")
print(score_por_variable.head())

print("Score total:")
print(df['score_total'].head())


Coeficientes del modelo:
edad_bin_WOE           -0.609283
ingreso_bin_WOE        -0.900489
historial_pagos_WOE    -0.223861
educacion_WOE          -0.844253
dependientes_bin_WOE   -0.357118
dtype: float64
Factor: 28.85390081777927
Ejemplo de score por variable:
   edad_bin_WOE  ingreso_bin_WOE  historial_pagos_WOE  educacion_WOE  \
0      2.020946         1.369644            -0.635477       2.760294   
1      2.020946        -9.784152             0.394905      -6.428740   
2      1.817705         1.369644             0.394905       2.760294   
3     -1.224749         1.369644            -0.555377       2.760294   
4      2.020946         1.369644            -0.555377       2.760294   

   dependientes_bin_WOE  
0             -1.055733  
1              0.301451  
2             -0.514120  
3              0.301451  
4              1.152180  
Score total:
0    604.459674
1    586.504411
2    605.828429
3    602.651262
4    606.747687
Name: score_total, dtype: float64
